# Phase 1: Data Foundation — Step 1.2: Data Validation

This notebook implements pandas assert-based validation checks on schemas, type consistency, value ranges, uniqueness of primary keys, and categorical constraints. Failures are captured in a pass/fail report rather than crashing the notebook.

In [1]:
import os
import pandas as pd

raw_dir = os.path.join("data", "raw")
print(f"Validating files in: {os.path.abspath(raw_dir)}")

Validating files in: C:\Users\Harshit Mishra\OneDrive\Desktop\enterprise_hr_ai\data\raw


## Helper Validation Function
We define `run_check` to run assertion functions and output `[PASS]`, `[FAIL]`, or `[ERROR]` messages without crashing the notebook.

In [2]:
def run_check(name, check_fn):
    try:
        check_fn()
        print(f"[PASS] {name}")
        return True
    except AssertionError as e:
        print(f"[FAIL] {name}: {e}")
        return False
    except Exception as e:
        print(f"[ERROR] {name}: {e}")
        return False

## 1. Validation of Employee Attrition (`employee_attrition.csv`)

In [3]:
df_attr = pd.read_csv(os.path.join(raw_dir, "employee_attrition.csv"))

# Schema Check
expected_attr_cols = {"Age", "Attrition", "EmployeeNumber", "JobRole", "EnvironmentSatisfaction", "JobSatisfaction", "RelationshipSatisfaction", "WorkLifeBalance"}
def check_attr_schema():
    missing = expected_attr_cols - set(df_attr.columns)
    assert not missing, f"Missing columns: {missing}"
run_check("employee_attrition.csv - Schema Check", check_attr_schema)

# PK Uniqueness Check
def check_attr_pk():
    assert df_attr["EmployeeNumber"].is_unique, "EmployeeNumber is not unique"
run_check("employee_attrition.csv - PK Uniqueness", check_attr_pk)

# Type Check
def check_attr_types():
    assert pd.api.types.is_integer_dtype(df_attr["Age"]), "Age is not an integer type"
    assert pd.api.types.is_integer_dtype(df_attr["EmployeeNumber"]), "EmployeeNumber is not an integer type"
run_check("employee_attrition.csv - Type Check", check_attr_types)

# Range Check (Age 18-100, Satisfaction/WorkLife 1-4 scale)
def check_attr_ranges():
    invalid_ages = df_attr[(df_attr["Age"] < 18) | (df_attr["Age"] > 100)]
    assert invalid_ages.empty, f"Found {len(invalid_ages)} rows with Age outside 18-100 range"
    
    for col in ["EnvironmentSatisfaction", "JobSatisfaction", "RelationshipSatisfaction", "WorkLifeBalance"]:
        invalid_scores = df_attr[~df_attr[col].isin([1, 2, 3, 4])]
        assert invalid_scores.empty, f"Column {col} has values outside [1, 2, 3, 4] range"
run_check("employee_attrition.csv - Range Check", check_attr_ranges)

# Category Check
def check_attr_categories():
    invalid_attr = df_attr[~df_attr["Attrition"].isin(["Yes", "No"])]
    assert invalid_attr.empty, f"Found unexpected Attrition categories: {df_attr['Attrition'].unique()}"
run_check("employee_attrition.csv - Category Check", check_attr_categories)

[PASS] employee_attrition.csv - Schema Check
[PASS] employee_attrition.csv - PK Uniqueness
[PASS] employee_attrition.csv - Type Check
[PASS] employee_attrition.csv - Range Check
[PASS] employee_attrition.csv - Category Check


## 2. Validation of HR Performance & Engagement (`hr_performance_engagement.csv`)

In [4]:
df_perf = pd.read_csv(os.path.join(raw_dir, "hr_performance_engagement.csv"))

# Schema Check
expected_perf_cols = {"Employee ID", "Age", "Engagement Score", "Satisfaction Score", "Work-Life Balance Score"}
def check_perf_schema():
    missing = expected_perf_cols - set(df_perf.columns)
    assert not missing, f"Missing columns: {missing}"
run_check("hr_performance_engagement.csv - Schema Check", check_perf_schema)

# PK Uniqueness Check
def check_perf_pk():
    assert df_perf["Employee ID"].is_unique, "Employee ID is not unique"
run_check("hr_performance_engagement.csv - PK Uniqueness", check_perf_pk)

# Type Check
def check_perf_types():
    assert pd.api.types.is_integer_dtype(df_perf["Age"]), "Age is not an integer type"
    assert pd.api.types.is_integer_dtype(df_perf["Employee ID"]), "Employee ID is not an integer type"
run_check("hr_performance_engagement.csv - Type Check", check_perf_types)

# Range Check (Age 18-100)
def check_perf_ranges():
    invalid_ages = df_perf[(df_perf["Age"] < 18) | (df_perf["Age"] > 100)]
    assert invalid_ages.empty, f"Found {len(invalid_ages)} rows with Age outside 18-100 range: {invalid_ages[['Employee ID', 'Age']].to_dict(orient='records')}"
run_check("hr_performance_engagement.csv - Age Range Check (18-100)", check_perf_ranges)

# Scores Range Check (Engagement/Satisfaction/WorkLife 1-5 scale based on observed values)
def check_perf_score_ranges():
    for col in ["Engagement Score", "Satisfaction Score", "Work-Life Balance Score"]:
        invalid_scores = df_perf[~df_perf[col].isin([1, 2, 3, 4, 5])]
        assert invalid_scores.empty, f"Column {col} has values outside [1, 2, 3, 4, 5] range"
run_check("hr_performance_engagement.csv - Score Ranges Check (1-5)", check_perf_score_ranges)

[PASS] hr_performance_engagement.csv - Schema Check
[PASS] hr_performance_engagement.csv - PK Uniqueness
[PASS] hr_performance_engagement.csv - Type Check
[FAIL] hr_performance_engagement.csv - Age Range Check (18-100): Found 2 rows with Age outside 18-100 range: [{'Employee ID': 1743, 'Age': 17}, {'Employee ID': 2038, 'Age': 17}]
[PASS] hr_performance_engagement.csv - Score Ranges Check (1-5)


## 3. Validation of Skills/Taxonomy (`occupation_data.csv`, `essential_skills.csv`, `software_skills.csv`)

In [5]:
df_occ = pd.read_csv(os.path.join(raw_dir, "occupation_data.csv"))
df_ess = pd.read_csv(os.path.join(raw_dir, "essential_skills.csv"))
df_soft = pd.read_csv(os.path.join(raw_dir, "software_skills.csv"))

# PK uniqueness checks
def check_occ_pk():
    assert df_occ["O*NET-SOC Code"].is_unique, "O*NET-SOC Code is not unique"
run_check("occupation_data.csv - PK Uniqueness", check_occ_pk)

# Schema checks
def check_skills_schemas():
    assert "O*NET-SOC Code" in df_ess.columns, "O*NET-SOC Code missing in essential_skills.csv"
    assert "Element ID" in df_ess.columns, "Element ID missing in essential_skills.csv"
    assert "O*NET-SOC Code" in df_soft.columns, "O*NET-SOC Code missing in software_skills.csv"
    assert "Element ID" in df_soft.columns, "Element ID missing in software_skills.csv"
run_check("Skills Datasets - Schema Check", check_skills_schemas)

[PASS] occupation_data.csv - PK Uniqueness
[PASS] Skills Datasets - Schema Check
